# ДЗ.10

Сгенерировать случайный сэмпл для отклика детектора (число фотоэлектронов на каждом ФЭУ) для сигнала определённой энергии в определённой точке пространства.
Использовать этот сэмпл как наблюдаемое (observed) для байесовского вывода: реконструкции энергии и положения события.

In [14]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt

R = 2.0
N = 40
r = 0.1
a = 0.5
Y = 10000

# Размещаем фотоумножители по параллелям, как в задаче 1
dots = []

for i in range(5):
    cos_theta = 1 - 2 * (i + 0.5) / 5
    sin_theta = np.sqrt(1 - cos_theta**2)

    for j in range(8):
        phi = 2 * np.pi * j / 8

        x = R * sin_theta * np.cos(phi)
        y = R * sin_theta * np.sin(phi)
        z = R * cos_theta

        dots.append([x, y, z])

centers = np.array(dots)

print(centers.shape)

(40, 3)


In [15]:
# Истинные параметры события
E_true = 1.0
coordinats_true = np.array([0.5, -0.4, 0.3])

# Расстояния от источника до фотоумножителей
d = centers - coordinats_true
lengths = np.linalg.norm(d, axis=1)

# Косинусы углов падения
cos_alpha = np.sum(d * centers, axis=1) / (R * lengths)

# Вероятности попадания на каждый фотокатод
p_true = r**2 * cos_alpha / (4 * lengths**2)

# Отклик детектора
rng = np.random.default_rng(42)

n_photons = rng.poisson(Y * E_true * p_true)
Q_obs = rng.poisson(a * n_photons)

print("Число фотоэлектронов на каждом ФЭУ:")
print(Q_obs)
print("Полный заряд:", np.sum(Q_obs))

Число фотоэлектронов на каждом ФЭУ:
[ 7  7  4  4  0  3  4  1  2  4  2  0  1  2  2  3  9  4  1  2  1  1  4 11
  3  1  6  1  1  6  2  6  2  2  0  0  3  4  6  2]
Полный заряд: 124


In [16]:
with pm.Model() as model:

    # Неизвестные параметры события
    E = pm.Uniform("E", lower=0.2, upper=2.0)

    x = pm.Uniform("x", lower=-1.0, upper=1.0)
    y = pm.Uniform("y", lower=-1.0, upper=1.0)
    z = pm.Uniform("z", lower=-1.0, upper=1.0)

    coordinats = pt.stack([x, y, z])

    # Расстояния до фотоумножителей
    d = centers - coordinats
    lengths = pt.sqrt(pt.sum(d**2, axis=1))

    # Вероятности попадания фотонов
    cos_alpha = pt.sum(d * centers, axis=1) / (R * lengths)
    p = r**2 * cos_alpha / (4 * lengths**2)

    # Среднее число фотоэлектронов на каждом ФЭУ
    mu = Y * E * p * a

    # Наблюдаемый отклик
    Q = pm.NegativeBinomial("Q", mu=mu, alpha=mu/a, observed=Q_obs)

with model:
    trace = pm.sample(
        1000,
        tune=1000,
        chains=2,
        cores=1,
        random_seed=42,
        progressbar=False
    )

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [E, x, y, z]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 2 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [17]:
print("Истинная энергия:", E_true)
print("Восстановленная энергия:", np.mean(trace.posterior["E"].values))

print("Истинные координаты:", coordinats_true)

print("Восстановленная координата x:", np.mean(trace.posterior["x"].values))
print("Восстановленная координата y:", np.mean(trace.posterior["y"].values))
print("Восстановленная координата z:", np.mean(trace.posterior["z"].values))

Истинная энергия: 1.0
Восстановленная энергия: 0.9959421305920653
Истинные координаты: [ 0.5 -0.4  0.3]
Восстановленная координата x: 0.57337437484763
Восстановленная координата y: -0.29222591144831706
Восстановленная координата z: 0.07138495797620814


In [18]:
# Находим интервалы для восстановленных параметров
for name in ["E", "x", "y", "z"]:
    values = trace.posterior[name].values.flatten()
    interval = np.percentile(values, [2.5, 97.5])

    print(name, ":", np.mean(values))
    print("95% интервал:", interval)


E : 0.9959421305920653
95% интервал: [0.79834415 1.20365343]
x : 0.57337437484763
95% интервал: [0.22671206 0.8868029 ]
y : -0.29222591144831706
95% интервал: [-0.65854193  0.07277798]
z : 0.07138495797620814
95% интервал: [-0.24113787  0.39591906]


по результатам проверки,видно, что энергия восстановилась достаточно точно, а для координат наблюдаются более заметные отклонения из-за статистических флуктуаций. При этом все истинные значения попали в соответствующие 95% интервалы.